# Retraining Gate Gap Handoff V1

This notebook consolidates the current retention gate, SOXL backtest reference baseline, current best replay model floor, and prior model-selection screens into a concrete handoff for the next retraining run. It is not a promotion artifact and does not retain a new actor.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'SPEC.md').exists():
        ROOT = parent
        break
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'retraining_gate_gap_handoff_v1'
CARD.mkdir(parents=True, exist_ok=True)

def load_json(path: str) -> dict:
    return json.loads((ROOT / path).read_text(encoding='utf-8'))

def as_float(value, default=0.0):
    if value is None or value == '':
        return default
    return float(value)

SOURCES = {
    'retention_gate': 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json',
    'retention_gate_csv': 'runtime/model_selection/card/model_retention_walkforward_gate_v1/retention_gate.csv',
    'checkpoint_drift_selector': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/summary.json',
    'checkpoint_selection_csv': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_selection_by_fold.csv',
    'checkpoint_candidates_csv': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_candidate_metrics.csv',
    'bc_ppo_drift_screen': 'runtime/model_selection/card/cap10_joint_contract_bc_ppo_drift_screen_v1/summary.json',
    'bc_slice_screen': 'runtime/model_selection/card/cap10_joint_contract_bc_slice_screen_v1/summary.json',
    'rebound_weighted_bc': 'runtime/model_selection/card/state_objective_rebound_weighted_bc_v1/summary.json',
    'rolling_heuristic_selector': 'runtime/model_selection/card/rolling_heuristic_selector_v1/summary.json',
    'heuristic_strategy_search': 'runtime/model_selection/card/heuristic_strategy_search_v1/summary.json',
}


In [2]:
retention = load_json(SOURCES['retention_gate'])
retention_rows = pd.read_csv(ROOT / SOURCES['retention_gate_csv'])
summary = retention['summary']

profile_rows = retention_rows[retention_rows['profile'] != 'soxl_backtest_policy_reimplementation_v1'].copy()
failure_counts = Counter()
for failures in profile_rows['failures'].fillna(''):
    for failure in str(failures).split(';'):
        if failure:
            failure_counts[failure] += 1

failure_summary = pd.DataFrame(
    [{'failure': key, 'count': value} for key, value in failure_counts.most_common()]
)
closest_by_wf_dd = (
    profile_rows.dropna(subset=['worst_wf_max_drawdown_pct'])
    .sort_values(['worst_wf_max_drawdown_pct', 'ytd_return_pct'], ascending=[True, False])
    .head(5)
)
closest_by_ytd = (
    profile_rows.sort_values(['ytd_return_pct', 'worst_wf_max_drawdown_pct'], ascending=[False, True])
    .head(5)
)

failure_summary.to_csv(CARD / 'retention_failure_summary.csv', index=False)
closest_by_wf_dd.to_csv(CARD / 'closest_by_walkforward_drawdown.csv', index=False)
closest_by_ytd.to_csv(CARD / 'closest_by_ytd_return.csv', index=False)

print('passing_profiles', summary['passing_profiles'])
print('top_failures', failure_summary.head(8).to_dict(orient='records'))
print('closest_by_wf_dd', closest_by_wf_dd[['profile','worst_wf_max_drawdown_pct','ytd_return_pct','failures']].to_dict(orient='records'))


passing_profiles []
top_failures [{'failure': 'ytd_return_below_best_alignment', 'count': 11}, {'failure': 'wf_max_drawdown_ge_20:wf_2020', 'count': 7}, {'failure': 'wf_max_drawdown_ge_20:wf_2023_2024', 'count': 6}, {'failure': 'wf_max_drawdown_ge_20:wf_2022', 'count': 5}, {'failure': 'missing_wf_window:wf_2020', 'count': 4}, {'failure': 'missing_wf_window:wf_2022', 'count': 4}, {'failure': 'missing_wf_window:wf_2023_2024', 'count': 4}, {'failure': 'ytd_return_below_soxlbacktest_baseline', 'count': 1}]
closest_by_wf_dd [{'profile': 'mode_conditioned_rebound_reward_train_v2', 'worst_wf_max_drawdown_pct': 21.83172, 'ytd_return_pct': 177.573569, 'failures': 'wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_ge_20:wf_2023_2024;ytd_return_below_best_alignment'}, {'profile': 'drawdown_rebound_reward_train_v1', 'worst_wf_max_drawdown_pct': 23.54683, 'ytd_return_pct': 172.832548, 'failures': 'wf_max_drawdown_ge_20:wf_2020;ytd_return_below_best_alignment'}, {'profile': 'state_conditioned_drawdown_r

In [3]:
checkpoint = load_json(SOURCES['checkpoint_drift_selector'])
bc_ppo = load_json(SOURCES['bc_ppo_drift_screen'])
bc_slice = load_json(SOURCES['bc_slice_screen'])
rebound = load_json(SOURCES['rebound_weighted_bc'])
rolling = load_json(SOURCES['rolling_heuristic_selector'])
heuristic = load_json(SOURCES['heuristic_strategy_search'])

checkpoint_selection = pd.read_csv(ROOT / SOURCES['checkpoint_selection_csv'])
checkpoint_candidates = pd.read_csv(ROOT / SOURCES['checkpoint_candidates_csv'])

source_cards = pd.DataFrame([
    {
        'source': 'model_retention_walkforward_gate_v1',
        'decision': 'no_replayable_profile_passes_retention_gate',
        'promotion_allowed': False,
        'use_for_next_training': True,
        'key_evidence': f"0/{summary['profiles_scanned']} replayable profiles pass; required YTD floor {summary['required_best_return_pct']:.6f}% and MaxDD < {summary['max_walkforward_drawdown_pct']:.2f}% across {','.join(summary['required_wf_windows'])}",
    },
    {
        'source': 'cap10_joint_contract_checkpoint_drift_selector_v1',
        'decision': checkpoint['selection']['decision'],
        'promotion_allowed': False,
        'use_for_next_training': True,
        'key_evidence': '; '.join(checkpoint['conclusion']),
    },
    {
        'source': 'cap10_joint_contract_bc_slice_screen_v1',
        'decision': bc_slice['decision'],
        'promotion_allowed': bool(bc_slice['promotion_allowed']),
        'use_for_next_training': bool(bc_slice.get('ppo_screen_allowed_next', False)),
        'key_evidence': f"selected_config={bc_slice['selected_config']}; min_fold_accuracy={bc_slice['selected_min_fold_accuracy']:.3f}; next={bc_slice['next_mechanism']}",
    },
    {
        'source': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
        'decision': bc_ppo['decision'],
        'promotion_allowed': False,
        'use_for_next_training': False,
        'key_evidence': '; '.join(bc_ppo['conclusion']),
    },
    {
        'source': 'state_objective_rebound_weighted_bc_v1',
        'decision': rebound['selection']['decision'],
        'promotion_allowed': bool(rebound['selection']['promotion_allowed']),
        'use_for_next_training': False,
        'key_evidence': '; '.join(rebound['conclusion']),
    },
    {
        'source': 'rolling_heuristic_selector_v1',
        'decision': rolling['decision'],
        'promotion_allowed': bool(rolling['promotion_allowed']),
        'use_for_next_training': False,
        'key_evidence': rolling['next_action'],
    },
])
source_cards.to_csv(CARD / 'source_card_decisions.csv', index=False)

checkpoint_selection.to_csv(CARD / 'selected_checkpoint_by_fold.csv', index=False)

print(source_cards[['source','decision','use_for_next_training']].to_string(index=False))
print(checkpoint_selection.to_string(index=False))


                                           source                                                               decision  use_for_next_training
              model_retention_walkforward_gate_v1                            no_replayable_profile_passes_retention_gate                   True
cap10_joint_contract_checkpoint_drift_selector_v1     checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion                   True
          cap10_joint_contract_bc_slice_screen_v1              joint_contract_bc_slice_ready_for_ppo_screen_no_promotion                   True
      cap10_joint_contract_bc_ppo_drift_screen_v1 joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion                  False
           state_objective_rebound_weighted_bc_v1                          rebound_weighting_no_improvement_no_promotion                  False
                    rolling_heuristic_selector_v1                                   hard_gate_no_rolling_selector_passes                

In [4]:
wf2022 = checkpoint_selection[checkpoint_selection['fold'].eq('wf_2022')].iloc[0].to_dict()
wf2023 = checkpoint_selection[checkpoint_selection['fold'].eq('wf_2023_2024')].iloc[0].to_dict()
ppo_wf2022 = checkpoint_candidates[
    checkpoint_candidates['fold'].eq('wf_2022') & checkpoint_candidates['candidate'].eq('ppo_selected_best')
].iloc[0].to_dict()

handoff = {
    'experiment': 'retraining_gate_gap_handoff_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'run_full_walkforward_cap10_joint_contract_bc_checkpoint_training',
    'promotion_allowed': False,
    'next_training_allowed': True,
    'canonical_reference_baseline': summary['reference_profile'],
    'current_best_ytd_return_pct': summary['current_best_return_pct'],
    'required_best_return_pct': summary['required_best_return_pct'],
    'required_soxlbacktest_return_pct': summary['reference_return_pct'],
    'retention_gate': {
        'max_walkforward_drawdown_pct': summary['max_walkforward_drawdown_pct'],
        'required_wf_windows': summary['required_wf_windows'],
        'passing_profile_count': summary['passing_profile_count'],
        'profiles_scanned': summary['profiles_scanned'],
    },
    'selected_training_source': 'cap10_joint_contract_checkpoint_drift_selector_v1',
    'start_from': {
        'candidate_family': 'cap10_joint_contract_bc_initialization_or_ppo_iter0_bc_equivalent',
        'wf_2022_selected_candidate': wf2022['selected_candidate'],
        'wf_2022_selected_relative_return': as_float(wf2022['selected_relative_return']),
        'wf_2022_selected_max_dd': as_float(wf2022['selected_max_dd']),
        'wf_2023_2024_selected_candidate': wf2023['selected_candidate'],
        'wf_2023_2024_selected_relative_return': as_float(wf2023['selected_relative_return']),
        'wf_2023_2024_selected_max_dd': as_float(wf2023['selected_max_dd']),
    },
    'do_not_repeat': [
        'do_not_promote_any_replayable_model_without_complete_wf_2020_wf_2022_wf_2023_2024_metrics',
        'do_not_continue_static_heuristic_or_rolling_selector_branch_without_model_replay_or_bc_candidate_paths',
        'do_not_lengthen_ppo_when_wf_2022_dd_delta_vs_bc_exceeds_2pp',
        'do_not_retry_rebound_weight_only_bc_without_target_enrichment_or_sampling_change',
    ],
    'required_next_run_controls': [
        'add_wf_2020_coverage_before_actor_retention',
        'use_bc_initialization_or_iter0_bc_equivalent_as_checkpoint_reference',
        'apply_checkpoint_drift_stop: ppo_vs_bc_max_dd_delta_pp <= 2.0',
        'apply_checkpoint_drift_stop: ppo_vs_bc_relative_delta_pp >= -5.0',
        'require_fold_gate_before_full_training_continuation',
        'export_daily_paths_for_dashboard_and_retention_gate_review',
    ],
    'nearest_known_gap': {
        'wf_2022_selected_max_dd_gap_to_15pct': round(max(0.0, as_float(wf2022['selected_max_dd']) - 15.0), 6),
        'wf_2022_selected_max_dd_gap_to_20pct': round(max(0.0, as_float(wf2022['selected_max_dd']) - 20.0), 6),
        'wf_2022_ppo_selected_max_dd_delta_vs_bc': round(as_float(ppo_wf2022['max_dd_delta_vs_bc']), 6),
        'wf_2022_ppo_selected_relative_delta_vs_bc': round(as_float(ppo_wf2022['relative_delta_vs_bc']), 6),
        'wf_2023_2024_selected_gate': bool(wf2023['selected_candidate_gate']),
    },
    'acceptance_criteria_for_retention': [
        'all required walk-forward windows present',
        'worst walk-forward MaxDD strictly below 20%',
        'YTD return >= soxl_backtest_policy_reimplementation_v1 YTD return',
        'YTD return >= 95% of current best replay model YTD return',
        'dashboard replay CSV includes date, strategy_nav, position, action, price/close fields',
    ],
    'sidecars': [
        'summary.json',
        'source_card_decisions.csv',
        'retention_failure_summary.csv',
        'closest_by_walkforward_drawdown.csv',
        'closest_by_ytd_return.csv',
        'selected_checkpoint_by_fold.csv',
        'training_objective_handoff.json',
        'conclusion.md',
    ],
}

(CARD / 'training_objective_handoff.json').write_text(json.dumps(handoff, indent=2, sort_keys=True) + '\n', encoding='utf-8')
summary_out = {
    'experiment': handoff['experiment'],
    'created_at': handoff['created_at'],
    'decision': handoff['decision'],
    'promotion_allowed': handoff['promotion_allowed'],
    'next_training_allowed': handoff['next_training_allowed'],
    'retention_gate': handoff['retention_gate'],
    'start_from': handoff['start_from'],
    'nearest_known_gap': handoff['nearest_known_gap'],
    'do_not_repeat': handoff['do_not_repeat'],
    'required_next_run_controls': handoff['required_next_run_controls'],
    'sidecars': handoff['sidecars'],
}
(CARD / 'summary.json').write_text(json.dumps(summary_out, indent=2, sort_keys=True) + '\n', encoding='utf-8')

lines = [
    '# Retraining Gate Gap Handoff V1',
    '',
    f"- Decision: `{handoff['decision']}`.",
    '- Promotion: blocked; this card does not retain or promote a new actor.',
    f"- Retention gate: 0/{summary['profiles_scanned']} replayable profiles pass complete WF + MaxDD/return alignment.",
    f"- SOXLBacktest reference: `{summary['reference_profile']}` YTD `{summary['reference_return_pct']:.6f}%`; current-best alignment floor `{summary['required_best_return_pct']:.6f}%`.",
    f"- Best available next source: cap10 joint-contract BC/checkpoint path. wf_2022 selected MaxDD `{handoff['start_from']['wf_2022_selected_max_dd']:.6f}%`, rel `{handoff['start_from']['wf_2022_selected_relative_return']:.6f}pp`; wf_2023_2024 selected MaxDD `{handoff['start_from']['wf_2023_2024_selected_max_dd']:.6f}%`, rel `{handoff['start_from']['wf_2023_2024_selected_relative_return']:.6f}pp`.",
    f"- Critical stop rule: do not lengthen PPO when wf_2022 MaxDD drift vs BC exceeds `2pp`; observed PPO-selected drift was `{handoff['nearest_known_gap']['wf_2022_ppo_selected_max_dd_delta_vs_bc']:.6f}pp`.",
    '- Next retraining run must add wf_2020 coverage before any actor retention, then rerun fixed-split + full walk-forward + dashboard replay export.',
]
(CARD / 'conclusion.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

print(json.dumps(summary_out, indent=2, sort_keys=True))


{
  "created_at": "2026-06-13T10:22:17.771588+00:00",
  "decision": "run_full_walkforward_cap10_joint_contract_bc_checkpoint_training",
  "do_not_repeat": [
    "do_not_promote_any_replayable_model_without_complete_wf_2020_wf_2022_wf_2023_2024_metrics",
    "do_not_continue_static_heuristic_or_rolling_selector_branch_without_model_replay_or_bc_candidate_paths",
    "do_not_lengthen_ppo_when_wf_2022_dd_delta_vs_bc_exceeds_2pp",
    "do_not_retry_rebound_weight_only_bc_without_target_enrichment_or_sampling_change"
  ],
  "experiment": "retraining_gate_gap_handoff_v1",
  "nearest_known_gap": {
    "wf_2022_ppo_selected_max_dd_delta_vs_bc": 18.820132,
    "wf_2022_ppo_selected_relative_delta_vs_bc": 5.351209,
    "wf_2022_selected_max_dd_gap_to_15pct": 3.455088,
    "wf_2022_selected_max_dd_gap_to_20pct": 0.0,
    "wf_2023_2024_selected_gate": true
  },
  "next_training_allowed": true,
  "promotion_allowed": false,
  "required_next_run_controls": [
    "add_wf_2020_coverage_before_actor_re